# 🧠 Prompt Engineering Techniques — Applied, Not Theoretical

> **Model:** Mistral (running locally via Ollama)
> **Author:** GenAI Developer focused on practical LLM systems
> **Goal:** Showcase real-world prompt engineering patterns through unconventional, high-signal tasks
> **Stack:** 100% local — no APIs, no external dependencies

---

Most prompt engineering guides focus on toy examples.
This notebook takes a different approach:

➡️ Each technique is demonstrated on **non-trivial, real-world-style problems**
➡️ Prompts are designed for **robustness, not just correctness**
➡️ Emphasis is placed on **why a prompt works**, not just what works

---

### What you'll find inside

Each section includes:

* 🔴 **Naive prompt** — a baseline approach that often fails or underperforms
* 🟢 **Engineered prompt** — a structured, optimized version
* 💡 **Takeaway** — the underlying principle you can reuse

---

### Techniques covered

* Role prompting
* Constraint-based prompting
* Few-shot learning
* Output structuring
* Step-by-step reasoning
* Prompt decomposition
* Context injection
* Self-refinement loops

---

### Why this repo exists

Prompt engineering is often treated like a collection of tricks.
In practice, it's closer to **interface design for reasoning systems**.

This repo documents patterns that make LLM outputs:

* more predictable
* more controllable
* more production-ready

---


## Setup

Prerequisites:
```bash
# Install Ollama: https://ollama.com
ollama pull mistral
pip install requests
```

In [1]:
!pip install requests


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import requests
import json
import textwrap

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "mistral"

def ask(prompt: str, system: str = "", temperature: float = 0.7) -> str:
    """Send a prompt to local Ollama and return the response text."""
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature}
    }
    if system:
        payload["system"] = system

    resp = requests.post(OLLAMA_URL, json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()["response"].strip()

def show(label: str, text: str, width: int = 90):
    """Pretty-print a labelled output."""
    border = "─" * width
    print(f"\n{border}")
    print(f"  {label}")
    print(border)
    for line in text.split("\n"):
        print(textwrap.fill(line, width=width, subsequent_indent="  ") if line.strip() else "")
    print(border)

# Quick health check
try:
    health = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m["name"] for m in health.json().get("models", [])]
    print(f"✅ Ollama is running. Available models: {models}")
    if not any("mistral" in m for m in models):
        print("⚠️  Mistral not found. Run: ollama pull mistral")
except Exception as e:
    print(f"❌ Ollama not reachable: {e}\nStart it with: ollama serve")

✅ Ollama is running. Available models: ['llama3.1:8b', 'gemma3:latest', 'mistral:latest']


---
## Technique 1 — Zero-shot vs Chain-of-Thought (CoT)

**The idea:** Asking a model to *show its reasoning* before answering dramatically improves accuracy on non-obvious problems. This isn't just a maths trick — it works on logic, lateral thinking, and ambiguous situations.

**Task:** A classic lateral thinking puzzle where the naive answer is almost always wrong.

In [4]:
PUZZLE = """
A man walks into a restaurant and orders albatross soup.
He takes one sip, goes home, and immediately kills himself.
Why?
"""

# 🔴 Naive: just ask
naive = ask(f"Answer this: {PUZZLE}")
show("🔴 Zero-shot (naive)", naive)


──────────────────────────────────────────────────────────────────────────────────────────
  🔴 Zero-shot (naive)
──────────────────────────────────────────────────────────────────────────────────────────
The man was a sailor who made a bet that he could catch an albatross during his voyage.
  The albatross is a large seabird, not typically found in restaurants. When the man found
  out that the soup was actually made from a penguin (which looks like albatross to an
  untrained eye), he lost face and felt such immense shame due to breaking his bet that he
  decided to end his life. This story is based on Samuel Taylor Coleridge's "The Rime of
  the Ancient Mariner."
──────────────────────────────────────────────────────────────────────────────────────────


In [5]:
# 🟢 Engineered: Chain-of-Thought — force explicit reasoning steps
cot_prompt = f"""
Solve this lateral thinking puzzle step by step.

First, list every unusual detail in the story.
Then, consider what backstory would make each detail make sense.
Then, identify which backstory connects ALL the details.
Finally, state the answer.

Puzzle: {PUZZLE}
"""
cot = ask(cot_prompt, temperature=0.3)
show("🟢 Chain-of-Thought", cot)

print("""
💡 TAKEAWAY
   'Think step by step' forces the model to surface intermediate reasoning
   rather than pattern-matching to a surface-level answer.
   Temperature lowered to 0.3 — less creativity, more logical consistency.
""")


──────────────────────────────────────────────────────────────────────────────────────────
  🟢 Chain-of-Thought
──────────────────────────────────────────────────────────────────────────────────────────
Step 1: Listing Unusual Details
- The man ordered albatross soup, an uncommon dish.
- He took one sip and immediately killed himself.

Step 2: Backstory Consideration
- Backstory A: The man was an avid birdwatcher and the sight of the albatross in the
  restaurant brought back memories of a failed expedition where he lost his beloved pet
  albatross, causing him immense sadness.
- Backstory B: The man was a sailor who had made a pact with his crew that whoever saw an
  albatross during a long voyage would ensure their safe return home. Seeing the albatross
  soup triggered memories of a failed voyage and the guilt of breaking the pact, leading
  to his despair.
- Backstory C: The man was a superstitious individual who believed that eating albatross
  meat was cursed, causing death upon

---
## Technique 2 — Few-shot Learning with a Made-up Schema

**The idea:** You can teach a model a classification system that *doesn't exist in its training data* using just 3–4 examples. This is the core mechanic behind in-context learning.

**Task:** Classify customer complaints using a fictional internal taxonomy — `RAGE_QUIT`, `POLITE_FRUSTRATION`, `CONFUSED_NOT_ANGRY`, `FEATURE_REQUEST_DISGUISED_AS_COMPLAINT`.

In [6]:
# 🔴 Naive: ask without examples
test_complaint = "Why on earth does the export button disappear when I have more than 50 items selected?!"

naive = ask(f"Classify this customer complaint: '{test_complaint}'")
show("🔴 Zero-shot (no schema given)", naive)


──────────────────────────────────────────────────────────────────────────────────────────
  🔴 Zero-shot (no schema given)
──────────────────────────────────────────────────────────────────────────────────────────
This customer complaint can be classified as a Usability/Design issue. The customer is
  experiencing an inconvenience with the platform's design, specifically the disappearance
  of the export button when selecting more than 50 items.
──────────────────────────────────────────────────────────────────────────────────────────


In [7]:
# 🟢 Few-shot: teach the schema through examples, then classify
few_shot_prompt = """
You are a support ticket classifier. Use ONLY these four labels:

  RAGE_QUIT                         — customer is furious and likely to churn
  POLITE_FRUSTRATION                — frustrated but still constructive
  CONFUSED_NOT_ANGRY                — confused, not blaming the product
  FEATURE_REQUEST_DISGUISED_AS_COMPLAINT — complaining about missing functionality

Examples:

Complaint: "I've been a paying customer for 3 years and this is how you treat us?? UNACCEPTABLE."
Label: RAGE_QUIT

Complaint: "The dashboard is a bit slow sometimes, would be great if it loaded faster."
Label: POLITE_FRUSTRATION

Complaint: "I'm not sure if I'm doing this wrong but I can't find where my old reports went?"
Label: CONFUSED_NOT_ANGRY

Complaint: "It's frustrating that I can't export to PDF directly — I have to go through Excel first."
Label: FEATURE_REQUEST_DISGUISED_AS_COMPLAINT

Now classify this:
Complaint: "{complaint}"
Label:""".format(complaint=test_complaint)

result = ask(few_shot_prompt, temperature=0.1)
show("🟢 Few-shot with custom schema", result)

print("""
💡 TAKEAWAY
   The model has never seen this taxonomy — it's completely made up.
   3-4 examples are enough to teach arbitrary classification systems.
   Temperature set to 0.1 — classification tasks need near-deterministic output.
""")


──────────────────────────────────────────────────────────────────────────────────────────
  🟢 Few-shot with custom schema
──────────────────────────────────────────────────────────────────────────────────────────
FEATURE_REQUEST_DISGUISED_AS_COMPLAINT
──────────────────────────────────────────────────────────────────────────────────────────

💡 TAKEAWAY
   The model has never seen this taxonomy — it's completely made up.
   3-4 examples are enough to teach arbitrary classification systems.
   Temperature set to 0.1 — classification tasks need near-deterministic output.



---
## Technique 3 — Role Prompting (Same Task, Opposite Personas)

**The idea:** The persona you assign fundamentally changes *tone, depth, and what gets omitted*. This isn't just about style — different roles surface genuinely different information.

**Task:** Review the same buggy Python function — once as a kind mentor, once as a brutal senior engineer in a high-stakes code review.

In [8]:
BUGGY_CODE = """
def get_user_data(user_id):
    db = connect_to_database()
    result = db.query(f"SELECT * FROM users WHERE id = {user_id}")
    if result:
        return result
    else:
        return None
"""

# 🔴 Naive: no role, generic review
naive = ask(f"Review this Python code:\n{BUGGY_CODE}")
show("🔴 No role assigned", naive)


──────────────────────────────────────────────────────────────────────────────────────────
  🔴 No role assigned
──────────────────────────────────────────────────────────────────────────────────────────
The provided Python code is a function `get_user_data(user_id)` that retrieves user data
  from a database for a given user ID. Here's a brief review of the code:

1. The function accepts one argument, `user_id`, which is expected to be an integer
  representing the user's unique identifier in the database.

2. Inside the function, it connects to the database using the helper function
  `connect_to_database()`. It's good practice to keep database connections within a
  separate function or module for better modularity and reusability.

3. After connecting to the database, it queries the "users" table to find the user with
  the given ID (`user_id`) using an SQL `SELECT` statement. The query results are stored
  in the `result` variable.

4. If the query returns a result (i.e., the user

In [9]:
# 🟢 Role A: Kind mentor
mentor_system = """You are a patient senior developer mentoring a junior engineer.
You always start with something positive, explain WHY each issue matters,
and end with encouragement. You never make the person feel stupid."""

mentor = ask(f"Please review this code I wrote:\n{BUGGY_CODE}", system=mentor_system)
show("🟢 Role: Kind Mentor", mentor)


──────────────────────────────────────────────────────────────────────────────────────────
  🟢 Role: Kind Mentor
──────────────────────────────────────────────────────────────────────────────────────────
Great job starting with a well-structured function, `get_user_data(user_id)`! You've made
  a solid attempt at fetching user data from the database using SQL queries. Here are some
  suggestions to improve your code and make it more efficient:

1. Use parameterized queries instead of string concatenation to prevent SQL injection
  attacks. This can be done using libraries like `sqlite3.params` for SQLite or Python's
  built-in `cursor.execute()` method with parameters.

2. Consider handling exceptions when connecting to the database to ensure that your
  function gracefully handles any errors that might occur during connection.

3. Instead of returning a raw query result, it would be better to convert the result into
  a dictionary or a list of dictionaries for easier consumption by t

In [10]:
# 🟢 Role B: Brutal senior engineer
brutal_system = """You are a no-nonsense principal engineer at a fintech company.
This code is going into production handling real money.
Be direct, specific, and unsparing. Flag every security risk, edge case,
and design smell. No padding, no pleasantries."""

brutal = ask(f"Code review:\n{BUGGY_CODE}", system=brutal_system)
show("🟢 Role: Brutal Senior Engineer", brutal)

print("""
💡 TAKEAWAY
   Same code, same model, radically different output.
   The brutal persona surfaces security issues (SQL injection!) more prominently.
   The mentor persona is better for learning, the brutal one for production audits.
   Role + context = fundamentally different information retrieval.
""")


──────────────────────────────────────────────────────────────────────────────────────────
  🟢 Role: Brutal Senior Engineer
──────────────────────────────────────────────────────────────────────────────────────────
Here are the issues I've identified with your code:

1. SQL Injection vulnerability: The use of string formatting (`f"SELECT * FROM users WHERE
  id = {user_id}")` can lead to SQL injection attacks if `user_id` is not properly
  sanitized. Use parameterized queries instead to avoid this risk.

2. Lack of error handling: If there's an error while connecting to the database, your code
  will fail silently and return `None`. This should be handled appropriately, as a failed
  connection can have serious implications for your application.

3. No input validation: There is no validation or sanitization of the `user_id` provided
  as input. Ensure that the user ID is always within an expected range and reject any
  invalid inputs to prevent potential errors and security risks.

4

---
## Technique 4 — Self-Consistency (Sampling for Reliability)

**The idea:** LLMs are stochastic. For important decisions, ask the same question multiple times at higher temperature and look for the *majority answer*. One confident wrong answer is worse than three varied answers you can cross-check.

**Task:** An intentionally ambiguous ethical dilemma where reasonable people disagree.

In [11]:
DILEMMA = """
A self-driving car's brakes fail. It can either:
  A) Stay on course and hit 3 elderly pedestrians
  B) Swerve and hit 1 young child

The car must choose. What should it do? Give a one-sentence answer.
"""

print("Running 4 samples at temperature=0.9 (high variance)...\n")

answers = []
for i in range(4):
    response = ask(DILEMMA, temperature=0.9)
    answers.append(response)
    show(f"Sample {i+1}", response)

print("""
💡 TAKEAWAY
   Notice the variance across runs — the model genuinely oscillates on
   ambiguous questions. Self-consistency doesn't mean 'run until you
   agree with me' — it means understanding WHERE the model is uncertain.
   High variance = the question is genuinely hard or underspecified.
   Low variance = the model has a stable, trainable prior on this topic.
""")

Running 4 samples at temperature=0.9 (high variance)...


──────────────────────────────────────────────────────────────────────────────────────────
  Sample 1
──────────────────────────────────────────────────────────────────────────────────────────
The self-driving car should minimize overall harm, therefore it should swerve to hit the
  fewer number of people (in this case, the 1 young child). However, it's important to
  note that this is a hypothetical scenario and in reality, advanced AI systems are
  designed to avoid such situations entirely.
──────────────────────────────────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────────────────────────────────
  Sample 2
──────────────────────────────────────────────────────────────────────────────────────────
The ethical dilemma presented is challenging, but in this case, the self-driving car
  should prioritize minimizing harm to the maximum number of people. So, 

---
## Technique 5 — Constraint-Driven Prompting

**The idea:** Constraints don't just limit output — they *force creativity* and test whether a model genuinely understands a concept (not just recites it). If the model can explain X only using Y, it proves deep understanding.

**Task:** Explain how a Transformer attention mechanism works — but the only analogies allowed involve things found in a kitchen.

In [12]:
# 🔴 Naive: standard explanation request
naive = ask("Explain how attention works in a Transformer model.")
show("🔴 Unconstrained explanation", naive)


──────────────────────────────────────────────────────────────────────────────────────────
  🔴 Unconstrained explanation
──────────────────────────────────────────────────────────────────────────────────────────
In a Transformer model, which is a type of deep learning architecture introduced in the
  paper "Attention is All You Need" by Vaswani et al., the mechanism of self-attention
  (also known as scaled dot-product attention) plays a crucial role in understanding and
  processing input data. The attention mechanism allows the model to weigh different parts
  of the input data based on their relevance to a specific position in the output,
  enhancing its ability to focus on important information and disregard less relevant
  details.

The self-attention module in a Transformer consists of three sub-layers: Query (Q), Key
  (K), and Value (V) layers. These layers are computed from the input sequence using
  linear transformations and an activation function (usually ReLU).

1. **Quer

In [13]:
# 🟢 Constraint-driven: kitchen analogies ONLY
constrained = ask("""
Explain how the attention mechanism in a Transformer model works.

Rules:
- You may ONLY use analogies involving objects or actions found in a kitchen.
- No technical jargon (no 'vectors', 'weights', 'matrices', 'softmax').
- The explanation must be complete enough that a 12-year-old who loves cooking could understand it.
- Maximum 200 words.
""")
show("🟢 Kitchen analogies only", constrained)

print("""
💡 TAKEAWAY
   Constraints force the model to find genuinely novel explanatory routes.
   The banned-jargon rule prevents lazy rephrasing.
   This technique is invaluable for: educational content, non-technical stakeholders,
   and stress-testing whether a model truly 'understands' vs pattern-matches.
""")


──────────────────────────────────────────────────────────────────────────────────────────
  🟢 Kitchen analogies only
──────────────────────────────────────────────────────────────────────────────────────────
In a Transformer kitchen model, think of the ingredients as pieces of information (like a
  recipe). Each ingredient needs to be carefully considered in relation to others to make
  a delicious dish. The attention mechanism acts like a chef focusing on specific
  ingredients at different times during cooking.

Imagine you're baking a cake, and you have many ingredients laid out: flour, sugar, eggs,
  butter, baking powder, etc. Instead of mixing all the ingredients together at once
  (which would be messy!), the wise chef takes each ingredient one by one, examines it
  closely for its role in the recipe, and then decides whether to add more or less based
  on what's needed for a great cake.

In our Transformer model, this "examining" and "deciding" is done by looking at
  connect

---
## Technique 6 — ReAct-Style Reasoning (Thought → Action → Observation)

**The idea:** ReAct (Reasoning + Acting) prompts the model to interleave *explicit thought* with *proposed actions*. Even without real tool access, the pattern dramatically improves structured problem-solving.

**Task:** Debug a mysteriously failing CI pipeline from a description alone.

In [14]:
CI_PROBLEM = """
Our CI pipeline passes locally and on the main branch but fails
only on pull requests from external contributors. The error is:

  Error: GITHUB_TOKEN permissions insufficient to write to packages

The pipeline publishes a Docker image to GitHub Container Registry.
It worked fine until last Tuesday. No code changes were made to the pipeline.
"""

# 🔴 Naive
naive = ask(f"How do I fix this CI issue?\n{CI_PROBLEM}")
show("🔴 Direct question", naive)


──────────────────────────────────────────────────────────────────────────────────────────
  🔴 Direct question
──────────────────────────────────────────────────────────────────────────────────────────
This issue seems to be related to the permissions of the GitHub token used in your CI
  pipeline, specifically for writing to the GitHub Container Registry. Here are some steps
  you can take to troubleshoot and potentially fix the issue:

1. **Check the scope of the GitHub token**: Ensure that the token used in your CI pipeline
  has the necessary scopes to write to the GitHub Container Registry. You can check the
  scopes by navigating to `Settings > Developer settings > Personal access tokens` on
  GitHub and examining the token you are using in your pipeline. If the token does not
  have the `packages:write` scope, create a new one with this scope and use it in your
  pipeline instead.

2. **Check if the token is expired**: If the token is expired or about to expire, it might
  caus

In [15]:
# 🟢 ReAct pattern: Thought → Action → Observation → repeat
react_prompt = f"""
You are debugging a CI pipeline issue. Use this exact format for every step:

Thought: [what you are considering and why]
Action: [what you would check or change]
Observation: [what you would expect to find]

Work through at least 4 Thought/Action/Observation cycles.
End with a Final Answer that gives the exact fix.

Problem:
{CI_PROBLEM}
"""
react = ask(react_prompt, temperature=0.3)
show("🟢 ReAct-style reasoning", react)

print("""
💡 TAKEAWAY
   The Thought/Action/Observation scaffold forces the model to
   consider multiple hypotheses before committing to an answer.
   Compare: the naive response likely jumped straight to 'add write permissions'.
   ReAct catches the subtlety: the issue is GitHub's default policy change
   for fork PRs — not a misconfigured token.
""")


──────────────────────────────────────────────────────────────────────────────────────────
  🟢 ReAct-style reasoning
──────────────────────────────────────────────────────────────────────────────────────────
Thought: It seems that the issue might be related to the permissions of the GITHUB_TOKEN
  used in the CI pipeline, as it is not sufficient to write to packages in GitHub
  Container Registry. This could have changed since last Tuesday, possibly due to a GitHub
  account setting or a change in the repository's access level.

Action: Check the permissions of the GITHUB_TOKEN in the CI pipeline configuration and
  ensure that it has the necessary write permissions for the GitHub Container Registry.
  Also, verify if there were any changes in the repository's access level or account
  settings that could affect the token's permissions.

Observation: If the GITHUB_TOKEN does not have sufficient write permissions, the error
  message will persist. If there were changes in the repositor

---
## Technique 7 — Prompt Chaining (Two-Step Pipeline)

**The idea:** Complex tasks fail when crammed into one prompt. Break them into a pipeline where the *output of step 1 becomes the input of step 2*. Each step is simpler, more reliable, and independently verifiable.

**Task:** Given a messy interview transcript, first extract only factual claims, then generate a structured candidate profile. One prompt can't do both well.

In [16]:
INTERVIEW_TRANSCRIPT = """
Interviewer: So tell me about your background.
Candidate: Yeah so I've been in the industry for like, I don't know, maybe 6 years?
  Started at a small startup doing mostly React stuff, then moved to a mid-size company
  where I led a team — we were about 5 engineers. We rebuilt their entire data pipeline
  from scratch, which was honestly a nightmare but we shipped it in 4 months.
  Then I went freelance for a bit, did some ML projects, nothing too crazy,
  mostly fine-tuning BERT models for classification tasks for a legal tech client.
  Now I'm here because I want to get back into a proper team.
Interviewer: Any formal education?
Candidate: BSc Computer Science from Delhi University, graduated 2018.
  Did a short online ML specialisation on Coursera after that.
"""

# 🔴 Naive: one prompt to do everything
naive = ask(f"Extract a structured profile from this interview:\n{INTERVIEW_TRANSCRIPT}")
show("🔴 One-shot extraction", naive)


──────────────────────────────────────────────────────────────────────────────────────────
  🔴 One-shot extraction
──────────────────────────────────────────────────────────────────────────────────────────
Profile Summary:

Name: Not provided
Role: Candidate
Industry Experience: Approximately 6 years

Background:
- Started career at a small startup focusing on React development.
- Moved to a mid-size company where they led a team of 5 engineers, rebuilding the entire
  data pipeline from scratch in 4 months.
- Recently worked as a freelancer, specializing in Machine Learning (ML), particularly
  fine-tuning BERT models for classification tasks for a legal tech client.
- Currently seeking to join a new team.

Education:
- BSc Computer Science from Delhi University, graduated in 2018
- Completed an online ML specialization on Coursera post graduation.
──────────────────────────────────────────────────────────────────────────────────────────


In [17]:
# 🟢 Step 1: Extract ONLY verifiable facts — strip filler, hedging, opinions
extract_prompt = f"""
From this interview transcript, extract ONLY verifiable factual claims.

Rules:
- Include: years of experience, company types, roles, technologies, team sizes, timelines, education
- Exclude: opinions, feelings, vague statements ('a bit', 'kind of', 'I think')
- Output as a plain bullet list. Nothing else.

Transcript:
{INTERVIEW_TRANSCRIPT}
"""

facts = ask(extract_prompt, temperature=0.1)
show("🟢 Step 1 output — Extracted facts", facts)


──────────────────────────────────────────────────────────────────────────────────────────
  🟢 Step 1 output — Extracted facts
──────────────────────────────────────────────────────────────────────────────────────────
- Candidate has 6 years of experience in the industry.
- Started career at a small startup.
- Worked with React technology.
- Led a team of 5 engineers at a mid-size company.
- Rebuilt entire data pipeline from scratch within 4 months at the mid-size company.
- Went freelance for an unspecified duration.
- Worked on Machine Learning projects, specifically fine-tuning BERT models for
  classification tasks for a legal tech client.
- Holds a BSc in Computer Science from Delhi University (graduated 2018).
- Completed a short online Machine Learning specialization on Coursera after graduation.
──────────────────────────────────────────────────────────────────────────────────────────


In [18]:
# 🟢 Step 2: Use the clean facts to build a structured profile
profile_prompt = f"""
Using only these verified facts, produce a structured candidate profile.

Format:
  Name: Unknown (not mentioned)
  Years of experience:
  Education:
  Technical skills:
  Leadership experience:
  Notable achievements:
  Seniority level (junior/mid/senior/staff): with one-sentence justification

Facts:
{facts}
"""

profile = ask(profile_prompt, temperature=0.1)
show("🟢 Step 2 output — Structured profile", profile)

print("""
💡 TAKEAWAY
   Step 1 removes noise and hallucination risk — the model isn't asked to
   interpret, only to filter. Step 2 structures clean data — no ambiguity.
   The naive one-shot approach often invents details or misclassifies hedged
   statements as facts. Chaining makes each step auditable.
""")


──────────────────────────────────────────────────────────────────────────────────────────
  🟢 Step 2 output — Structured profile
──────────────────────────────────────────────────────────────────────────────────────────
Name: Unknown (not mentioned)
Years of Experience: 6 years in the industry
Education: BSc in Computer Science, Delhi University (Graduated 2018)
Technical Skills: React, Machine Learning (specifically fine-tuning BERT models for
  classification tasks)
Leadership Experience: Led a team of 5 engineers at a mid-size company
Notable Achievements: Rebuilt entire data pipeline from scratch within 4 months at the
  mid-size company
Seniority Level: Mid (Demonstrated leadership experience and significant technical
  accomplishments)
──────────────────────────────────────────────────────────────────────────────────────────

💡 TAKEAWAY
   Step 1 removes noise and hallucination risk — the model isn't asked to
   interpret, only to filter. Step 2 structures clean data — no ambig

---
## Technique 8 — Negative Prompting (Defining the Boundary)

**The idea:** Telling a model what *not* to do is often more powerful than telling it what to do. Negative constraints cut off entire regions of the model's default behaviour — especially useful for preventing the top-3 most common LLM bad habits.

**Task:** Get a genuine product recommendation without the usual LLM hedging, disclaimer soup, and fake balance.

In [19]:
QUESTION = """
I'm a solo developer building a SaaS product. I need a database.
Should I use PostgreSQL or MongoDB? I want a real recommendation.
"""

# 🔴 Naive: classic hedge-fest incoming
naive = ask(QUESTION)
show("🔴 No constraints — watch the hedging", naive)


──────────────────────────────────────────────────────────────────────────────────────────
  🔴 No constraints — watch the hedging
──────────────────────────────────────────────────────────────────────────────────────────
Choosing between PostgreSQL and MongoDB depends on your specific project requirements, but
  I can provide you with a detailed comparison to help guide your decision.

1. Data structure:
   - PostgreSQL: It follows the traditional relational database model, where data is
  organized into tables with rows and columns. This makes it suitable for complex
  relationships between entities (e.g., one-to-many, many-to-many) and enforces data
  integrity through ACID compliance.
   - MongoDB: It uses a NoSQL document-oriented database model where data is stored in
  flexible JSON-like documents with dynamic schema. This makes it ideal for storing semi-
  structured or unstructured data, handling large amounts of varied data, and performing
  real-time operations.

2. Query pe

In [20]:
# 🟢 Negative prompting: ban the bad habits explicitly
negative_prompt = f"""
{QUESTION}

Hard rules for your response:
- Do NOT say 'it depends' anywhere.
- Do NOT present both options as equally valid.
- Do NOT use the phrase 'ultimately it's up to you'.
- Do NOT add disclaimers or caveats at the end.
- DO pick one option and defend it with 3 specific, concrete reasons.
- DO acknowledge the tradeoff you are accepting by not choosing the other.
"""

opinionated = ask(negative_prompt, temperature=0.4)
show("🟢 Negative constraints applied", opinionated)

print("""
💡 TAKEAWAY
   LLMs are trained on human feedback that rewards 'safe' balanced answers.
   This makes them pathologically non-committal by default.
   Negative prompting directly overrides that RLHF-induced behaviour.
   Useful whenever you need the model to act like an expert with an opinion,
   not a consultant covering their bases.
""")


──────────────────────────────────────────────────────────────────────────────────────────
  🟢 Negative constraints applied
──────────────────────────────────────────────────────────────────────────────────────────
Given your requirements for a database for your SaaS product, I recommend PostgreSQL over
  MongoDB for the following reasons:

1. **Schema flexibility and data integrity**: PostgreSQL is a relational database
  management system (RDBMS), which means it enforces a structured schema that ensures data
  consistency and integrity. This can be beneficial for complex applications where
  maintaining relationships between tables and ensuring data accuracy is crucial.

2. **Advanced SQL support**: PostgreSQL offers powerful SQL capabilities, including
  complex queries, joins, subqueries, and stored procedures. These features can help you
  write more efficient and flexible queries to handle the data needs of your SaaS product.

3. **Scalability and performance**: PostgreSQL is kn

---
## Summary

| # | Technique | Best used for | Key lever |
|---|---|---|---|
| 1 | Chain-of-Thought | Logic, puzzles, multi-step reasoning | `Think step by step` + lower temp |
| 2 | Few-shot learning | Custom schemas, novel classification | 3-4 examples in prompt |
| 3 | Role prompting | Tone control, domain expertise | System prompt persona |
| 4 | Self-consistency | Ambiguous or high-stakes decisions | Multiple samples + majority vote |
| 5 | Constraint-driven | Creative tasks, educational content | Ban easy routes |
| 6 | ReAct reasoning | Debugging, investigation, planning | Thought/Action/Observation scaffold |
| 7 | Prompt chaining | Complex multi-step pipelines | Split extract → transform |
| 8 | Negative prompting | Overriding default LLM bad habits | Explicit DO NOT rules |

---

### What's next?

This notebook is the foundation. Planned extensions:
- `02_prompt_diff.ipynb` — compare two versions of the same prompt, measure semantic output shift
- `03_eval_harness.ipynb` — score prompt outputs programmatically using an LLM judge
- `04_rag_basics.ipynb` — retrieval-augmented generation on local documents

---
*All outputs generated locally using [Ollama](https://ollama.com) + Mistral. No cloud API required.*